In [1]:
# CSV to parquet file
# This notebook can run only in AWS glue notebooks


Welcome to the Glue Interactive Sessions Kernel
For more information on available magic commands, please type %help in any new cell.

Please view our Getting Started page to access the most up-to-date information on the Interactive Sessions kernel: https://docs.aws.amazon.com/glue/latest/dg/interactive-sessions.html
Installed kernel version: 1.0.10 


In [1]:
import sys
from awsglue.transforms import *
from awsglue.utils import getResolvedOptions
from pyspark.context import SparkContext
from awsglue.context import GlueContext
from awsglue.job import Job
from pyspark.sql.functions import to_timestamp, col
from pyspark.sql.types import IntegerType, FloatType, StringType
from awsglue.dynamicframe import DynamicFrame


sc = SparkContext.getOrCreate()
glueContext = GlueContext(sc)
spark = glueContext.spark_session
job = Job(glueContext)

Trying to create a Glue session for the kernel.
Session Type: glueetl
Session ID: c558245d-285b-421a-87d6-706ecbe67cb0
Applying the following default arguments:
--glue_kernel_version 1.0.10
--enable-glue-datacatalog true
Waiting for session c558245d-285b-421a-87d6-706ecbe67cb0 to get into ready status...
Session c558245d-285b-421a-87d6-706ecbe67cb0 has been created.



In [2]:
# Read Data in CSV From AWS S3

In [2]:
wearable_df = glueContext.create_dynamic_frame_from_options(
    connection_type='s3',
    connection_options={
        "paths": ["s3://parquet-glue--bucket-1/merged_wearable_data_AWS_Test.csv"]
    },
    format="csv",
    format_options={
        "withHeader": True
    }
)

In [3]:
# Display schema and first 5 records

In [4]:
wearable_df.printSchema()
wearable_df.toDF().show(5)  

root
|-- timestamp: string
|-- customer_id: string
|-- patient_name: string
|-- heart_rate_bpm: string
|-- spo2_pct: string
|-- steps: string
|-- skin_temp_c: string
|-- hrv_ms: string
|-- respiratory_rate: string
|-- activity: string

+----------------+--------------------+------------+--------------+--------+-----+-----------+------+----------------+--------+
|       timestamp|         customer_id|patient_name|heart_rate_bpm|spo2_pct|steps|skin_temp_c|hrv_ms|respiratory_rate|activity|
+----------------+--------------------+------------+--------------+--------+-----+-----------+------+----------------+--------+
|11/04/2026 00:05|CUST_amara_patel_...| Amara Patel|            94|    94.1|   47|       36.5|    52|              12| walking|
|11/04/2026 01:45|CUST_amara_patel_...| Amara Patel|            98|    95.7|  229|       36.9|    71|              12| walking|
|11/04/2026 02:44|CUST_amara_patel_...| Amara Patel|            88|    95.1|  110|       37.3|    41|              13| resti

In [ ]:
# Remap data types to correct data type

In [5]:
# Convert to Spark DataFrame first
df = wearable_df.toDF()

# Cast columns to correct data types
df_cleaned = df \
    .withColumn("timestamp",        to_timestamp(col("timestamp"), "dd/MM/yyyy HH:mm")) \
    .withColumn("heart_rate_bpm",   col("heart_rate_bpm").cast(IntegerType())) \
    .withColumn("spo2_pct",         col("spo2_pct").cast(FloatType())) \
    .withColumn("steps",            col("steps").cast(IntegerType())) \
    .withColumn("skin_temp_c",      col("skin_temp_c").cast(FloatType())) \
    .withColumn("hrv_ms",           col("hrv_ms").cast(IntegerType())) \
    .withColumn("respiratory_rate", col("respiratory_rate").cast(IntegerType()))
    # customer_id, patient_name, activity stay as StringType (already correct)

# Verify
df_cleaned.printSchema()
df_cleaned.show(5)

root
 |-- timestamp: timestamp (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- patient_name: string (nullable = true)
 |-- heart_rate_bpm: integer (nullable = true)
 |-- spo2_pct: float (nullable = true)
 |-- steps: integer (nullable = true)
 |-- skin_temp_c: float (nullable = true)
 |-- hrv_ms: integer (nullable = true)
 |-- respiratory_rate: integer (nullable = true)
 |-- activity: string (nullable = true)

+-------------------+--------------------+------------+--------------+--------+-----+-----------+------+----------------+--------+
|          timestamp|         customer_id|patient_name|heart_rate_bpm|spo2_pct|steps|skin_temp_c|hrv_ms|respiratory_rate|activity|
+-------------------+--------------------+------------+--------------+--------+-----+-----------+------+----------------+--------+
|2026-04-11 00:05:00|CUST_amara_patel_...| Amara Patel|            94|    94.1|   47|       36.5|    52|              12| walking|
|2026-04-11 01:45:00|CUST_amara_patel_...| Am

In [4]:
# Write To AWS S3 and create Table in Glue Catalog

In [12]:
# Convert Spark DataFrame to DynamicFrame FIRST
dynamic_cleaned = DynamicFrame.fromDF(df_cleaned, glueContext, "dynamic_cleaned")

# Write to S3
write_S3_parquet = glueContext.getSink(
    path="s3://parquet-glue--bucket-1/output/wearable_parquet/",
    connection_type="s3",
    updateBehavior="UPDATE_IN_DATABASE",
    partitionKeys=["customer_id"],
    compression="gzip",
    enableUpdateCatalog=True,
    transformation_ctx="write_S3_parquet"
)

# Set Catalog info
write_S3_parquet.setCatalogInfo(
    catalogDatabase="wearable",
    catalogTableName="customers"
)

# Set format and write
write_S3_parquet.setFormat("glueparquet")
write_S3_parquet.writeFrame(dynamic_cleaned)  # must be DynamicFrame not df_cleaned
                               